In [1]:
import torch
print(torch.cuda.is_available())

False


In [2]:
import SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader=SoccerNetDownloader(LocalDirectory="path/to/SoccerNetGS")

In [3]:
# downloading validatation data
mySoccerNetDownloader.downloadDataTask(task="gamestate-2024", split=["valid"])

In [7]:
import zipfile
# replace filepath with your specific filepath
path = "path/to/SoccerNetGS/gamestate-2024/valid.zip"
with zipfile.ZipFile(path, "r") as zip_ref:
    # this takes all json files where we have our labels
    json_files = [json for json in zip_ref.namelist() if json.endswith(".json")]

    print(f"Found {len(json_files)} labeled json files")
    print(f"Sample: {json_files[:5]}")

    # extracting labeled files
    zip_ref.extractall(path="data/raw/", members = json_files)

Found 59 labeled json files
Sample: ['SNGS-033/Labels-GameState.json', 'SNGS-052/Labels-GameState.json', 'SNGS-091/Labels-GameState.json', 'SNGS-032/Labels-GameState.json', 'SNGS-078/Labels-GameState.json']


In [8]:
# inspecting label dataset
import json

with open("data/raw/SNGS-033/Labels-GameState.json", "r") as f:
    data = json.load(f)

print("Root keys:", list(data.keys()))
for k in data.keys():
    if isinstance(data[k], list):
        print(f"  '{k}': list with {len(data[k])} items")
    elif isinstance(data[k], dict):
        print(f"  '{k}': dict with keys {list(data[k].keys())}")
    else:
        print(f"  '{k}': {type(data[k]).__name__} = {data[k]}")

Root keys: ['info', 'images', 'annotations', 'categories']
  'info': dict with keys ['version', 'game_id', 'id', 'num_tracklets', 'action_position', 'action_class', 'visibility', 'game_time_start', 'game_time_stop', 'clip_start', 'clip_stop', 'name', 'im_dir', 'frame_rate', 'seq_length', 'im_ext']
  'images': list with 750 items
  'annotations': list with 9427 items
  'categories': list with 7 items


In [9]:
# Look at the first annotation entry
sample_item = data["annotations"][0]

# Pretty-print just that single dictionary
print(json.dumps(sample_item, indent=2))

{
  "id": "2033000001",
  "image_id": "2033000001",
  "track_id": 1,
  "supercategory": "object",
  "category_id": 3,
  "attributes": {
    "role": "referee",
    "jersey": null,
    "team": null
  },
  "bbox_image": {
    "x": 1441,
    "y": 798,
    "x_center": 1472.0,
    "y_center": 868.0,
    "w": 62,
    "h": 140
  },
  "bbox_pitch": {
    "x_bottom_left": 17.94676853811122,
    "y_bottom_left": 34.2566329130484,
    "x_bottom_right": 18.61722546005316,
    "y_bottom_right": 34.691501543313194,
    "x_bottom_middle": 18.2822622968887,
    "y_bottom_middle": 34.475176364049915
  },
  "bbox_pitch_raw": {
    "x_bottom_left": 18.378800225740804,
    "y_bottom_left": 34.13199405481367,
    "x_bottom_right": 19.04465313885331,
    "y_bottom_right": 34.567908270471165,
    "x_bottom_middle": 18.71196452007346,
    "y_bottom_middle": 34.35119730208327
  }
}


In [17]:
"""
This cell saves necessary data for YOLO box positioning to build a 3d to 2d map
projection model
"""
import polars as pl
import json
from pathlib import Path

records = []
path = Path("data/raw/")
pattern = "*SNGS*/*.json"
for file_path in path.glob(pattern):
    if file_path.is_file():
        print(f"Opening file: {file_path.name}")
        with open(file_path, "r") as f:
            data = json.load(f)
            for ann in data["annotations"]:
                # what we want the dictionary to look like in polars
                if "bbox_image" not in ann or "track_id" not in ann:
                    continue
                pitch_info = ann.get("bbox_pitch")
                record = {
                    "clip_id": data.get("info", {}).get("name"),
                    "image_id": ann["image_id"],
                    "track_id": ann.get("track_id"),
                    "role": ann.get("attributes", {}).get("role"),
                    "team": ann.get("attributes", {}).get("team"),
                    "jersey": ann.get("attributes", {}).get("jersey"),
                    "bbox_x1": ann["bbox_image"]["x"],
                    "bbox_y1": ann["bbox_image"]["y"],
                    "bbox_x2": ann["bbox_image"]["x"] + ann["bbox_image"]["w"],
                    "bbox_y2": ann["bbox_image"]["y"] + ann["bbox_image"]["h"],
                    "pitch_x_m": ann["bbox_pitch"]["x_bottom_middle"]
                    if pitch_info
                    else None,
                    "pitch_y_m": ann["bbox_pitch"]["y_bottom_middle"]
                    if pitch_info
                    else None,
                }
                records.append(record)
                
df = pl.DataFrame(records)

print(df.shape)
print(df["clip_id"].n_unique())

Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameState.json
Opening file: Labels-GameSta

In [18]:
df.head()

clip_id,image_id,track_id,role,team,jersey,bbox_x1,bbox_y1,bbox_x2,bbox_y2,pitch_x_m,pitch_y_m
str,str,i64,str,str,str,i64,i64,i64,i64,f64,f64
"""SNGS-039""","""2039000001""",1,"""player""","""right""","""23""",377,723,418,844,-40.157759,33.871538
"""SNGS-039""","""2039000001""",2,"""goalkeeper""","""left""",null,1285,385,1313,467,-49.690928,3.592799
"""SNGS-039""","""2039000001""",3,"""player""","""right""","""20""",359,593,427,690,-47.02089,28.830346
"""SNGS-039""","""2039000001""",4,"""referee""",null,null,1618,624,1657,734,-30.917567,16.404741
"""SNGS-039""","""2039000001""",5,"""player""","""left""","""36""",508,597,566,688,-45.894067,27.366738


In [21]:
# writing to parquet files

from pathlib import Path
Path("data/processed").mkdir(parents=True, exist_ok=True)

df.write_parquet("data/processed/annotations.parquet")

In [22]:
print(df["role"].value_counts())

shape: (5, 2)
┌────────────┬────────┐
│ role       ┆ count  │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ goalkeeper ┆ 26020  │
│ other      ┆ 399    │
│ ball       ┆ 40900  │
│ referee    ┆ 61847  │
│ player     ┆ 619500 │
└────────────┴────────┘


In [24]:
print(df.filter(pl.col("role") == "player")["team"].value_counts())

shape: (2, 2)
┌───────┬────────┐
│ team  ┆ count  │
│ ---   ┆ ---    │
│ str   ┆ u32    │
╞═══════╪════════╡
│ left  ┆ 306888 │
│ right ┆ 312612 │
└───────┴────────┘


In [25]:
print(
    df.select(
        pl.col("bbox_x1").min().alias("min_x1"),
        pl.col("bbox_x2").max().alias("max_x2"),
        pl.col("bbox_y1").min().alias("min_y1"),
        pl.col("bbox_y2").max().alias("max_y2"),
    )
)

shape: (1, 4)
┌────────┬────────┬────────┬────────┐
│ min_x1 ┆ max_x2 ┆ min_y1 ┆ max_y2 │
│ ---    ┆ ---    ┆ ---    ┆ ---    │
│ i64    ┆ i64    ┆ i64    ┆ i64    │
╞════════╪════════╪════════╪════════╡
│ 0      ┆ 1920   ┆ 0      ┆ 1080   │
└────────┴────────┴────────┴────────┘
